# 데이터 가져오기

## 독립변수

In [1]:
import requests
import pandas as pd
import urllib3

# SSL 경고 메시지 숨기기
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# API 키 입력
API_KEY = "5STQAATD94Z76TI7W683"

# ECOS open API로 데이터 가져오기
def get_ecos_data(stat_code, cycle, start_date, end_date, item_code, col_name):
    url = (
        f"https://ecos.bok.or.kr/api/StatisticSearch/"
        f"{API_KEY}/json/kr/1/3000/"                                   # 1~3000번째 데이터: 주말, 공휴일 제외 10년치 약 3000개
        f"{stat_code}/{cycle}/{start_date}/{end_date}/{item_code}"     # 통계표 코드/주기(D:일별, M:월별)/시작일/종료일/항목 코드
    )
    # 요청 보내기
    response = requests.get(url, verify=False)
    data = response.json()

    # 데이터 파싱
    rows = data["StatisticSearch"]["row"]
    df = pd.DataFrame(rows)

    # 필요한 컬럼만 정리
    df = df[["TIME", "DATA_VALUE"]].rename(columns={
        "TIME": "날짜",
        "DATA_VALUE": col_name
    })

    df[col_name] = pd.to_numeric(df[col_name])  # 숫자 타입으로 변환

    return df

1. 원/달러 환율, 한국 기준금리

In [2]:
ex_rate = get_ecos_data("731Y001", "D", "20190101", "20241231", "0000001", "원/달러환율")
kr_rate = get_ecos_data("722Y001", "M", "201901", "202412", "0101000", "한국금리")

print(ex_rate.shape)
print(ex_rate.head(10))

(1483, 2)
         날짜  원/달러환율
0  20190102  1118.1
1  20190103  1119.0
2  20190104  1127.5
3  20190107  1125.2
4  20190108  1118.0
5  20190109  1121.7
6  20190110  1121.6
7  20190111  1118.9
8  20190114  1117.6
9  20190115  1121.4


In [3]:
print(kr_rate.shape)
print(kr_rate.head(10))

(72, 2)
       날짜  한국금리
0  201901  1.75
1  201902  1.75
2  201903  1.75
3  201904  1.75
4  201905  1.75
5  201906  1.75
6  201907  1.50
7  201908  1.50
8  201909  1.50
9  201910  1.25


2. 미국연방기금금리(FFR)
- 단위: %

In [4]:
us_rate = pd.read_csv("./data/FEDFUNDS.csv")

print(us_rate.shape)
print(us_rate.head(10))

(72, 2)
  observation_date  FEDFUNDS
0       2019-01-01      2.40
1       2019-02-01      2.40
2       2019-03-01      2.41
3       2019-04-01      2.42
4       2019-05-01      2.39
5       2019-06-01      2.38
6       2019-07-01      2.40
7       2019-08-01      2.13
8       2019-09-01      2.04
9       2019-10-01      1.83


In [5]:
us_rate.columns = ['날짜', '미국금리']
us_rate.head(3)

,날짜,미국금리
0,2019-01-01,2.40
1,2019-02-01,2.40
2,2019-03-01,2.41


3. 유가

In [6]:
oil_price = pd.read_csv("./data/국제_원유가격20190101-20241231.csv")

oil_price = oil_price.rename(columns={"기간": "날짜"})

#wti   = oil_price[["날짜", "WTI"]].copy()
dubai = oil_price[["날짜", "Dubai"]].copy()

# print(wti.head(3))
print(dubai.head(3))

          날짜  Dubai
0  19년01월02일  51.86
1  19년01월03일  53.20
2  19년01월04일  55.59


## 종속변수

1. 품목별 국가별 수출입 실적

In [63]:
import glob

# 폴더 경로 지정
folder = r"C:\Users\926174\Downloads\수출입 실적"

files = glob.glob(folder + r"\수출입 실적(품목별+국가별)_*.csv")

trade_list = []
for file in sorted(files):
    df = pd.read_csv(file, encoding='utf-8-sig')
    trade_list.append(df)
    print(f"{file} 로드 완료 → {df.shape}")

# 하나로 합치기
trade = pd.concat(trade_list, ignore_index=True)

# 총계 행 제거
trade = trade[trade['기간'] != '총계'].reset_index(drop=True)

print(f"\n최종 shape: {trade.shape}")
trade.head(5)

C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_2015.csv 로드 완료 → (95804, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_2016.csv 로드 완료 → (96814, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_2017.csv 로드 완료 → (97724, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_2018.csv 로드 완료 → (99286, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_201901_11.csv 로드 완료 → (92203, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_201912.csv 로드 완료 → (8458, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_2020.csv 로드 완료 → (99617, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_202101_11.csv 로드 완료 → (93486, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_202112.csv 로드 완료 → (8593, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_202201_11.csv 로드 완료 → (92691, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_202212.csv 로드 완료 → (8477, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_202301_11.csv 로드 완료 → (93452, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_2023

,기간,국가,HS코드,품목명,수출 중량,수출 금액,수입 중량,수입 금액,무역수지
0,2015-01,가나,3.0,어류ㆍ갑각류ㆍ연체동물과 그 밖의 수생(水生) 무척추동물,81.0,120,0.0,0,120
1,2015-01,가나,12.0,"채유(採油)에 적합한 종자와 과실, 각종 종자와 과실, 공업용ㆍ의약용 식물, 짚과 ...",0.0,2,0.0,0,2
2,2015-01,가나,15.0,"동물성ㆍ식물성ㆍ미생물성 지방과 기름 및이들의 분해생산물, 조제한 식용 지방과 동물성...",0.0,0,17.5,56,-56
3,2015-01,가나,17.0,당류(糖類)와 설탕과자,1.5,33,0.0,0,33
4,2015-01,가나,18.0,코코아와 그 조제품,0.8,28,1223.8,1980,-1952


In [10]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import time

# ================================================
# STEP 1. 기본 설정
# ================================================

API_KEY = "d14e288e04bef1da401d66602694b68e03f685e96b078e003b3091eeb8bb8210"   # ← Decoding 키 입력

START_YEAR  = 2019
END_YEAR    = 2024
OUTPUT_FILE = "trade_data_final.csv"
API_URL     = "http://apis.data.go.kr/1220000/nitemtrade/getNitemtradeList"


# ================================================
# STEP 2. 수집할 HS코드 50개
# ================================================

HS_CODES = {
    # --- 환율 민감 (15개) ---
    "854231": ("반도체_메모리",       "환율"),
    "854232": ("반도체_비메모리",     "환율"),
    "854239": ("기타집적회로",        "환율"),
    "851762": ("통신기기부품",        "환율"),
    "851769": ("기타통신기기",        "환율"),
    "852872": ("평판디스플레이",      "환율"),
    "870321": ("승용차_1000cc이하",   "환율"),
    "870322": ("승용차_1000~1500cc", "환율"),
    "870323": ("승용차_1500~3000cc", "환율"),
    "870324": ("승용차_3000cc초과",   "환율"),
    "870840": ("자동차변속기",        "환율"),
    "870850": ("자동차구동축",        "환율"),
    "870899": ("기타자동차부품",      "환율"),
    "901380": ("광학기기",            "환율"),
    "903289": ("계측제어기기",        "환율"),
    # --- 유가 민감 (15개) ---
    "271012": ("경유등유",            "유가"),
    "271019": ("기타석유제품",        "유가"),
    "271111": ("LPG액화",             "유가"),
    "271121": ("LNG액화",             "유가"),
    "290110": ("에틸렌",              "유가"),
    "290121": ("에틸렌기타",          "유가"),
    "290129": ("기타기초유분",        "유가"),
    "390110": ("폴리에틸렌저밀도",    "유가"),
    "390120": ("폴리에틸렌고밀도",    "유가"),
    "390210": ("폴리프로필렌",        "유가"),
    "390410": ("PVC",                 "유가"),
    "390760": ("PET",                 "유가"),
    "330499": ("화장품기타",          "유가"),
    "320419": ("합성염료",            "유가"),
    "381400": ("혼합용제희석제",      "유가"),
    # --- 금리 민감 (10개) ---
    "847989": ("일반기계기타",        "금리"),
    "847950": ("산업용로봇",          "금리"),
    "842951": ("불도저굴착기",        "금리"),
    "842952": ("스크레이퍼",          "금리"),
    "890190": ("선박기타",            "금리"),
    "890120": ("탱커",                "금리"),
    "890130": ("냉동선",              "금리"),
    "720826": ("열연강판",            "금리"),
    "720839": ("냉연강판",            "금리"),
    "730511": ("강관",                "금리"),
    # --- 복합 충격 (10개) ---
    "850760": ("리튬이온이차전지",    "복합"),
    "850780": ("기타이차전지",        "복합"),
    "848620": ("반도체제조장비",      "복합"),
    "848630": ("반도체공정장비",      "복합"),
    "300490": ("의약품기타",          "복합"),
    "300420": ("항생제",              "복합"),
    "880240": ("항공기",              "복합"),
    "880330": ("항공기부품",          "복합"),
    "854140": ("태양전지",            "복합"),
    "902780": ("분석기기",            "복합"),
}


# ================================================
# STEP 3. 수출 대상국 25개
# ================================================

COUNTRIES = {
    "US": "미국",       "CN": "중국",       "VN": "베트남",
    "HK": "홍콩",       "JP": "일본",       "TW": "대만",
    "IN": "인도",       "SG": "싱가포르",   "MY": "말레이시아",
    "AU": "호주",       "DE": "독일",       "NL": "네덜란드",
    "GB": "영국",       "PL": "폴란드",     "IT": "이탈리아",
    "SA": "사우디",     "AE": "UAE",        "MX": "멕시코",
    "CA": "캐나다",     "BR": "브라질",     "TH": "태국",
    "ID": "인도네시아", "PH": "필리핀",     "BD": "방글라데시",
    "RU": "러시아",
}


# ================================================
# STEP 4. API 호출 함수 (XML 파싱)
# ================================================

def get_one(hs_code, country_code, year):
    """
    XML 응답을 받아서 파싱하는 함수
    """

    params = {
        "serviceKey": API_KEY,
        "strtYymm":   f"{year}01",
        "endYymm":    f"{year}12",
        "hsSgn":      hs_code,
        "cntyCd":     country_code,
    }

    try:
        resp = requests.get(API_URL, params=params, timeout=15)
        resp.raise_for_status()

        # XML 파싱
        root  = ET.fromstring(resp.text)
        items = root.findall(".//item")

        result = []
        for item in items:
            year_val = item.findtext("year", "")

            # "총계" 행 제거
            if year_val == "총계":
                continue

            result.append({
                "year":        year_val,
                "expDlr":      item.findtext("expDlr",      "0"),
                "expWgt":      item.findtext("expWgt",      "0"),
                "impDlr":      item.findtext("impDlr",      "0"),
                "impWgt":      item.findtext("impWgt",      "0"),
                "balPayments": item.findtext("balPayments", "0"),
            })

        return result

    except ET.ParseError as e:
        # XML 파싱 실패 시 응답 내용 출력 (디버깅용)
        print(f"\n  ⚠️  XML 파싱 오류 [{hs_code}][{country_code}] {year}년")
        print(f"      응답 내용 (앞 200자): {resp.text[:200]}")
        return []

    except Exception as e:
        print(f"\n  ⚠️  오류 [{hs_code}][{country_code}] {year}년: {e}")
        return []


# ================================================
# STEP 5. 전체 수집
# ================================================

def collect():

    result = []
    total  = len(HS_CODES) * len(COUNTRIES) * (END_YEAR - START_YEAR + 1)
    count  = 0

    for hs_code, (hs_name, shock_type) in HS_CODES.items():
        print(f"\n▶ [{hs_code}] {hs_name} ({shock_type})")

        for country_code, country_name in COUNTRIES.items():
            for year in range(START_YEAR, END_YEAR + 1):

                count += 1
                print(f"  {country_name}({country_code}) {year}년 ({count}/{total})", end=" ")

                items = get_one(hs_code, country_code, year)

                for item in items:
                    year_month = item["year"].replace(".", "-")  # "2019.01" → "2019-01"

                    result.append({
                        "연월":     year_month,
                        "연도":     year_month[:4],
                        "월":       year_month[5:7],
                        "HS코드":   hs_code,
                        "품목명":   hs_name,
                        "충격유형": shock_type,
                        "국가코드": country_code,
                        "국가명":   country_name,
                        "수출금액": item["expDlr"],
                        "수출중량": item["expWgt"],
                        "수입금액": item["impDlr"],
                        "수입중량": item["impWgt"],
                        "무역수지": item["balPayments"],
                    })

                print(f"→ {len(items)}건")
                time.sleep(0.2)

    return result


# ================================================
# STEP 6. 저장
# ================================================

def save(result):

    df = pd.DataFrame(result)

    for col in ["수출금액", "수출중량", "수입금액", "수입중량", "무역수지"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    df["날짜"] = pd.to_datetime(df["연월"], format="%Y-%m", errors="coerce")
    df = df.sort_values(["충격유형", "HS코드", "국가코드", "날짜"]).reset_index(drop=True)
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\n{'='*55}")
    print(f"✅ 저장 완료: {OUTPUT_FILE}")
    print(f"   총 행수:  {len(df):,}행")
    print(f"   기간:     {df['연월'].min()} ~ {df['연월'].max()}")
    print(f"\n▶ 충격유형별 행수:")
    print(df.groupby("충격유형")["연월"].count().to_string())
    print(f"\n▶ 미리보기 (상위 5행):")
    print(df.head().to_string(index=False))

    return df


# ================================================
# 실행
# ================================================

if __name__ == "__main__":

    total_calls = len(HS_CODES) * len(COUNTRIES) * (END_YEAR - START_YEAR + 1)

    print("=" * 55)
    print("관세청 수출입 데이터 수집")
    print(f"HS코드:  {len(HS_CODES)}개")
    print(f"국가:    {len(COUNTRIES)}개국")
    print(f"기간:    {START_YEAR}년 ~ {END_YEAR}년")
    print(f"총 호출: {total_calls:,}번 (트래픽 {total_calls/10000*100:.0f}% 사용)")
    print("=" * 55)

    result = collect()

    if result:
        df = save(result)
    else:
        print("⚠️ 수집된 데이터가 없습니다. API 키와 파라미터를 확인하세요.")

관세청 수출입 데이터 수집
HS코드:  50개
국가:    25개국
기간:    2019년 ~ 2024년
총 호출: 7,500번 (트래픽 75% 사용)

▶ [854231] 반도체_메모리 (환율)
  미국(US) 2019년 (1/7500) → 78건
  미국(US) 2020년 (2/7500) → 73건
  미국(US) 2021년 (3/7500) → 74건
  미국(US) 2022년 (4/7500) → 48건
  미국(US) 2023년 (5/7500) → 48건
  미국(US) 2024년 (6/7500) → 48건
  중국(CN) 2019년 (7/7500) → 88건
  중국(CN) 2020년 (8/7500) → 89건
  중국(CN) 2021년 (9/7500) → 90건
  중국(CN) 2022년 (10/7500) → 50건
  중국(CN) 2023년 (11/7500) → 48건
  중국(CN) 2024년 (12/7500) → 48건
  베트남(VN) 2019년 (13/7500) → 63건
  베트남(VN) 2020년 (14/7500) → 78건
  베트남(VN) 2021년 (15/7500) → 67건
  베트남(VN) 2022년 (16/7500) → 43건
  베트남(VN) 2023년 (17/7500) → 39건
  베트남(VN) 2024년 (18/7500) → 45건
  홍콩(HK) 2019년 (19/7500) → 76건
  홍콩(HK) 2020년 (20/7500) → 75건
  홍콩(HK) 2021년 (21/7500) → 74건
  홍콩(HK) 2022년 (22/7500) → 48건
  홍콩(HK) 2023년 (23/7500) → 48건
  홍콩(HK) 2024년 (24/7500) → 48건
  일본(JP) 2019년 (25/7500) → 66건
  일본(JP) 2020년 (26/7500) → 63건
  일본(JP) 2021년 (27/7500) → 66건
  일본(JP) 2022년 (28/7500) → 48건
  일본(JP) 2023년 (29/7500) 

- 오류 항목 다시 불러오고 병합

In [11]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import time
import os

# ================================================
# STEP 1. 기본 설정
# ================================================

API_KEY        = "d14e288e04bef1da401d66602694b68e03f685e96b078e003b3091eeb8bb8210"
API_URL        = "http://apis.data.go.kr/1220000/nitemtrade/getNitemtradeList"

ORIGINAL_FILE  = "trade_data_final.csv"   # 기존 수집 파일
RETRY_FILE     = "trade_data_retry.csv"   # 재수집 결과
MERGED_FILE    = "trade_data_merged.csv"  # 최종 병합 결과

# HS코드 → (품목명, 충격유형) 매핑
HS_INFO = {
    "854231": ("반도체_메모리",       "환율"),
    "854232": ("반도체_비메모리",     "환율"),
    "854239": ("기타집적회로",        "환율"),
    "851762": ("통신기기부품",        "환율"),
    "851769": ("기타통신기기",        "환율"),
    "852872": ("평판디스플레이",      "환율"),
    "870321": ("승용차_1000cc이하",   "환율"),
    "870322": ("승용차_1000~1500cc", "환율"),
    "870323": ("승용차_1500~3000cc", "환율"),
    "870324": ("승용차_3000cc초과",   "환율"),
    "870840": ("자동차변속기",        "환율"),
    "870850": ("자동차구동축",        "환율"),
    "870899": ("기타자동차부품",      "환율"),
    "901380": ("광학기기",            "환율"),
    "903289": ("계측제어기기",        "환율"),
    "271012": ("경유등유",            "유가"),
    "271019": ("기타석유제품",        "유가"),
    "271111": ("LPG액화",             "유가"),
    "271121": ("LNG액화",             "유가"),
    "290110": ("에틸렌",              "유가"),
    "290121": ("에틸렌기타",          "유가"),
    "290129": ("기타기초유분",        "유가"),
    "390110": ("폴리에틸렌저밀도",    "유가"),
    "390120": ("폴리에틸렌고밀도",    "유가"),
    "390210": ("폴리프로필렌",        "유가"),
    "390410": ("PVC",                 "유가"),
    "390761": ("PET",                 "유가"),   # 390760 → 390761 교체
    "330499": ("화장품기타",          "유가"),
    "320419": ("합성염료",            "유가"),
    "381400": ("혼합용제희석제",      "유가"),
    "847989": ("일반기계기타",        "금리"),
    "847950": ("산업용로봇",          "금리"),
    "842951": ("불도저굴착기",        "금리"),
    "842952": ("스크레이퍼",          "금리"),
    "890190": ("선박기타",            "금리"),
    "890120": ("탱커",                "금리"),
    "890130": ("냉동선",              "금리"),
    "720826": ("열연강판",            "금리"),
    "720839": ("냉연강판",            "금리"),
    "730511": ("강관",                "금리"),
    "850760": ("리튬이온이차전지",    "복합"),
    "850780": ("기타이차전지",        "복합"),
    "848620": ("반도체제조장비",      "복합"),
    "848630": ("반도체공정장비",      "복합"),
    "300490": ("의약품기타",          "복합"),
    "300420": ("항생제",              "복합"),
    "880240": ("항공기",              "복합"),
    "880330": ("항공기부품",          "복합"),
    "854140": ("태양전지",            "복합"),
    "902780": ("분석기기",            "복합"),
}

COUNTRY_INFO = {
    "US": "미국",       "CN": "중국",       "VN": "베트남",
    "HK": "홍콩",       "JP": "일본",       "TW": "대만",
    "IN": "인도",       "SG": "싱가포르",   "MY": "말레이시아",
    "AU": "호주",       "DE": "독일",       "NL": "네덜란드",
    "GB": "영국",       "PL": "폴란드",     "IT": "이탈리아",
    "SA": "사우디",     "AE": "UAE",        "MX": "멕시코",
    "CA": "캐나다",     "BR": "브라질",     "TH": "태국",
    "ID": "인도네시아", "PH": "필리핀",     "BD": "방글라데시",
    "RU": "러시아",
}


# ================================================
# STEP 2. 재수집 대상
# ================================================

# ① 타임아웃 오류 110건
TIMEOUT_TARGETS = [
    ("851769", "SA", 2021), ("852872", "DE", 2021), ("852872", "GB", 2023),
    ("870321", "US", 2019), ("870322", "US", 2019), ("870323", "JP", 2021),
    ("870323", "AU", 2024), ("870324", "VN", 2023), ("870840", "CN", 2021),
    ("870840", "TW", 2020), ("870840", "DE", 2024), ("870850", "IN", 2022),
    ("870850", "SA", 2023), ("870850", "ID", 2021), ("870850", "BD", 2020),
    ("870899", "TW", 2019), ("870899", "TW", 2020), ("901380", "VN", 2023),
    ("271012", "DE", 2023), ("271012", "IT", 2020), ("271012", "AE", 2019),
    ("271012", "AE", 2024), ("271019", "JP", 2019), ("271019", "JP", 2023),
    ("271019", "SG", 2023), ("271019", "AU", 2022), ("271019", "NL", 2023),
    ("271111", "NL", 2020), ("271121", "MY", 2024), ("290110", "NL", 2023),
    ("290121", "PL", 2023), ("290121", "MX", 2022), ("290129", "CN", 2019),
    ("290129", "DE", 2022), ("290129", "CA", 2023), ("390110", "AU", 2024),
    ("390110", "MX", 2022), ("390120", "DE", 2019), ("390210", "MY", 2023),
    ("390210", "IT", 2021), ("390410", "SG", 2022), ("390410", "BD", 2019),
    ("390760", "US", 2022), ("390760", "SG", 2024), ("390760", "GB", 2023),
    ("390760", "RU", 2019), ("330499", "US", 2021), ("330499", "MY", 2020),
    ("330499", "AE", 2023), ("320419", "IT", 2021), ("381400", "DE", 2023),
    ("381400", "RU", 2022), ("847989", "NL", 2023), ("847950", "CN", 2021),
    ("847950", "MY", 2022), ("847950", "CA", 2024), ("847950", "BD", 2019),
    ("842951", "PL", 2021), ("842951", "ID", 2024), ("842952", "MX", 2019),
    ("842952", "CA", 2024), ("890190", "TW", 2021), ("890190", "AE", 2019),
    ("890190", "AE", 2021), ("890120", "TW", 2020), ("890130", "US", 2020),
    ("890130", "CN", 2019), ("890130", "DE", 2022), ("890130", "CA", 2023),
    ("720826", "US", 2020), ("720826", "VN", 2023), ("720826", "IN", 2024),
    ("720826", "AU", 2021), ("720826", "GB", 2022), ("720826", "CA", 2020),
    ("720839", "HK", 2020), ("720839", "AU", 2021), ("720839", "GB", 2022),
    ("720839", "AE", 2023), ("720839", "ID", 2021), ("720839", "RU", 2021),
    ("730511", "BD", 2021), ("850760", "JP", 2019), ("850780", "DE", 2020),
    ("850780", "IT", 2023), ("848620", "CN", 2022), ("848620", "NL", 2023),
    ("848620", "PL", 2019), ("848620", "ID", 2019), ("848630", "AU", 2021),
    ("848630", "DE", 2023), ("848630", "NL", 2022), ("848630", "IT", 2019),
    ("848630", "IT", 2024), ("848630", "AE", 2020), ("848630", "BD", 2021),
    ("300490", "US", 2024), ("300490", "MY", 2022), ("300490", "BD", 2024),
    ("300420", "CN", 2023), ("300420", "PL", 2019), ("300420", "ID", 2019),
    ("880240", "IT", 2024), ("880330", "CN", 2022), ("880330", "TW", 2020),
    ("880330", "SA", 2020), ("880330", "BR", 2022), ("854140", "RU", 2024),
    ("902780", "BR", 2022), ("902780", "RU", 2023),
]

# ② PET 교체 (390760 → 390761) 전체 25개국 x 6년
COUNTRIES = list(COUNTRY_INFO.keys())
PET_TARGETS = [
    ("390761", c, y)
    for c in COUNTRIES
    for y in range(2019, 2025)
]

# 전체 재수집 대상 합치기
ALL_TARGETS = TIMEOUT_TARGETS + PET_TARGETS

print(f"타임아웃 오류: {len(TIMEOUT_TARGETS)}건")
print(f"PET 교체 수집: {len(PET_TARGETS)}건")
print(f"총 재수집 대상: {len(ALL_TARGETS)}건")


# ================================================
# STEP 3. API 호출 함수
# ================================================

def get_one(hs_code, country_code, year, max_retry=3):
    params = {
        "serviceKey": API_KEY,
        "strtYymm":   f"{year}01",
        "endYymm":    f"{year}12",
        "hsSgn":      hs_code,
        "cntyCd":     country_code,
    }

    for attempt in range(1, max_retry + 1):
        try:
            resp = requests.get(API_URL, params=params, timeout=15)
            resp.raise_for_status()
            root  = ET.fromstring(resp.text)
            items = root.findall(".//item")
            result = []
            for item in items:
                year_val = item.findtext("year", "")
                if year_val == "총계":
                    continue
                result.append({
                    "year":        year_val,
                    "expDlr":      item.findtext("expDlr",      "0"),
                    "expWgt":      item.findtext("expWgt",      "0"),
                    "impDlr":      item.findtext("impDlr",      "0"),
                    "impWgt":      item.findtext("impWgt",      "0"),
                    "balPayments": item.findtext("balPayments", "0"),
                })
            return result

        except ET.ParseError:
            print(f"  ⚠️  XML 파싱 오류: {resp.text[:100]}")
            return []

        except Exception as e:
            if attempt < max_retry:
                wait = attempt * 5
                print(f"  ⚠️  재시도 {attempt}/{max_retry} → {wait}초 대기...")
                time.sleep(wait)
            else:
                print(f"  ❌  최종 실패: {e}")
                return []


# ================================================
# STEP 4. 재수집 실행
# ================================================

def retry_collect():
    result = []
    total  = len(ALL_TARGETS)

    print(f"\n{'='*50}")
    print(f"재수집 시작 (총 {total}건)")
    print(f"{'='*50}")

    for i, (hs_code, country_code, year) in enumerate(ALL_TARGETS, 1):
        hs_name, shock_type = HS_INFO.get(hs_code, ("미분류", "기타"))
        country_name        = COUNTRY_INFO.get(country_code, country_code)

        # 구분선 출력 (타임아웃→PET 전환 시)
        if i == len(TIMEOUT_TARGETS) + 1:
            print(f"\n{'='*50}")
            print("▶ PET (390761) 신규 수집 시작")
            print(f"{'='*50}")

        print(f"  ({i}/{total}) [{hs_code}] {country_name}({country_code}) {year}년", end=" ")

        items = get_one(hs_code, country_code, year)

        for item in items:
            year_month = item["year"].replace(".", "-")
            result.append({
                "연월":     year_month,
                "연도":     year_month[:4],
                "월":       year_month[5:7],
                "HS코드":   hs_code,
                "품목명":   hs_name,
                "충격유형": shock_type,
                "국가코드": country_code,
                "국가명":   country_name,
                "수출금액": item["expDlr"],
                "수출중량": item["expWgt"],
                "수입금액": item["impDlr"],
                "수입중량": item["impWgt"],
                "무역수지": item["balPayments"],
            })

        print(f"→ {len(items)}건")
        time.sleep(0.3)

    return pd.DataFrame(result)


# ================================================
# STEP 5. 병합
# ================================================

def merge(df_retry):

    if not os.path.exists(ORIGINAL_FILE):
        print(f"⚠️  기존 파일({ORIGINAL_FILE})이 없습니다.")
        return df_retry

    df_original = pd.read_csv(ORIGINAL_FILE, dtype=str)
    print(f"\n기존 데이터:   {len(df_original):,}행")
    print(f"재수집 데이터: {len(df_retry):,}행")

    # 기존 데이터에서 390760(PET 구버전) 제거
    before = len(df_original)
    df_original = df_original[df_original["HS코드"] != "390760"]
    removed = before - len(df_original)
    if removed > 0:
        print(f"기존 390760(PET 구버전) 제거: {removed}행")

    # 재수집 데이터 타입 맞추기
    df_retry = df_retry.astype(str)

    # 병합
    df_merged = pd.concat([df_original, df_retry], ignore_index=True)

    # 중복 제거 (연월+HS코드+국가코드 기준, 재수집 우선)
    before = len(df_merged)
    df_merged = df_merged.drop_duplicates(
        subset=["연월", "HS코드", "국가코드"], keep="last"
    )
    print(f"중복 제거:     {before - len(df_merged)}행")

    # 숫자형 변환 및 정렬
    for col in ["수출금액", "수출중량", "수입금액", "수입중량", "무역수지"]:
        df_merged[col] = pd.to_numeric(df_merged[col], errors="coerce").fillna(0)

    df_merged["날짜"] = pd.to_datetime(df_merged["연월"], format="%Y-%m", errors="coerce")
    df_merged = df_merged.sort_values(
        ["충격유형", "HS코드", "국가코드", "날짜"]
    ).reset_index(drop=True)

    df_merged.to_csv(MERGED_FILE, index=False, encoding="utf-8-sig")

    print(f"\n{'='*50}")
    print(f"✅ 병합 완료: {MERGED_FILE}")
    print(f"   최종 행수: {len(df_merged):,}행")
    print(f"\n▶ HS코드별 행수 (상위 10개):")
    print(df_merged.groupby(["HS코드","품목명"])["연월"].count()
          .sort_values(ascending=False).head(10).to_string())

    return df_merged


# ================================================
# 실행
# ================================================

if __name__ == "__main__":

    print("=" * 50)
    print("재수집 및 병합 시작")
    print(f"  타임아웃 오류: {len(TIMEOUT_TARGETS)}건")
    print(f"  PET 교체 수집: {len(PET_TARGETS)}건 (390760→390761)")
    print(f"  총 API 호출:   {len(ALL_TARGETS)}건")
    print("=" * 50)

    # 재수집
    df_retry = retry_collect()

    # 재수집 결과 저장
    if not df_retry.empty:
        df_retry.to_csv(RETRY_FILE, index=False, encoding="utf-8-sig")
        print(f"\n재수집 결과 저장: {RETRY_FILE} ({len(df_retry):,}행)")

    # 병합
    print(f"\n{'='*50}")
    print("기존 데이터와 병합 중...")
    df_merged = merge(df_retry)

타임아웃 오류: 110건
PET 교체 수집: 150건
총 재수집 대상: 260건
재수집 및 병합 시작
  타임아웃 오류: 110건
  PET 교체 수집: 150건 (390760→390761)
  총 API 호출:   260건

재수집 시작 (총 260건)
  (1/260) [851769] 사우디(SA) 2021년 → 8건
  (2/260) [852872] 독일(DE) 2021년 → 31건
  (3/260) [852872] 영국(GB) 2023년 → 24건
  (4/260) [870321] 미국(US) 2019년 → 9건
  (5/260) [870322] 미국(US) 2019년 → 21건
  (6/260) [870323] 일본(JP) 2021년 → 48건
  (7/260) [870323] 호주(AU) 2024년 → 24건
  (8/260) [870324] 베트남(VN) 2023년 → 4건
  (9/260) [870840] 중국(CN) 2021년 → 12건
  (10/260) [870840] 대만(TW) 2020년 → 12건
  (11/260) [870840] 독일(DE) 2024년 → 12건
  (12/260) [870850] 인도(IN) 2022년 → 24건
  (13/260) [870850] 사우디(SA) 2023년 → 4건
  (14/260) [870850] 인도네시아(ID) 2021년 → 24건
  (15/260) [870850] 방글라데시(BD) 2020년 → 4건
  (16/260) [870899] 대만(TW) 2019년 → 28건
  (17/260) [870899] 대만(TW) 2020년 → 24건
  (18/260) [901380] 베트남(VN) 2023년 → 24건
  (19/260) [271012] 독일(DE) 2023년 → 17건
  (20/260) [271012] 이탈리아(IT) 2020년 → 8건
  (21/260) [271012] UAE(AE) 2019년 → 17건
  (22/260) [271012] UAE(AE) 2024년 → 18건


In [12]:
import pandas as pd
df = pd.read_csv("trade_data_final.csv", dtype=str)
print(df[["연월", "HS코드", "국가코드"]].head(20).to_string())
print("\n중복 행수:", df.duplicated(subset=["연월","HS코드","국가코드"]).sum())

         연월    HS코드 국가코드
0   2020-11  720826   AU
1   2020-12  720826   AU
2   2023-07  720826   AU
3   2023-10  720826   AU
4   2023-11  720826   AU
5   2023-12  720826   AU
6   2024-03  720826   AU
7   2024-05  720826   AU
8   2024-08  720826   AU
9   2024-12  720826   AU
10  2019-01  720826   BD
11  2019-10  720826   BD
12  2020-01  720826   BD
13  2020-02  720826   BD
14  2020-03  720826   BD
15  2020-03  720826   BD
16  2020-06  720826   BD
17  2020-07  720826   BD
18  2020-12  720826   BD
19  2021-02  720826   BD

중복 행수: 123272


In [13]:
import pandas as pd

# 1. 기존 파일 로드
df = pd.read_csv("trade_data_final.csv", dtype=str)
print(f"로드: {len(df):,}행")

# 2. 기존 데이터 내 중복 먼저 제거 (첫 번째 행 유지)
df = df.drop_duplicates(subset=["연월", "HS코드", "국가코드"], keep="first")
print(f"중복 제거 후: {len(df):,}행")

# 3. 390760 제거
df = df[df["HS코드"] != "390760"]
print(f"390760 제거 후: {len(df):,}행")

# 4. 재수집 데이터 로드 및 병합
retry = pd.read_csv("trade_data_retry.csv", dtype=str)
df_merged = pd.concat([df, retry], ignore_index=True)

# 5. 최종 중복 제거 (재수집 우선)
df_merged = df_merged.drop_duplicates(subset=["연월", "HS코드", "국가코드"], keep="last")
print(f"최종: {len(df_merged):,}행")

# 6. 저장
df_merged.to_csv("trade_data_merged.csv", index=False, encoding="utf-8-sig")
print("✅ 저장 완료")

로드: 184,661행
중복 제거 후: 61,389행
390760 제거 후: 61,389행
최종: 63,576행
✅ 저장 완료


In [16]:
# PET 확인
import pandas as pd
df = pd.read_csv("trade_data_merged.csv", dtype=str)
pet = df[df["HS코드"] == "390761"]
print(f"PET 행수: {len(pet)}")
print(pet.groupby("국가명")["연월"].count().to_string())

PET 행수: 1370
국가명
UAE      11
네덜란드     51
대만       72
독일       71
러시아      52
말레이시아    72
멕시코      63
미국       72
방글라데시     4
베트남      72
브라질      50
사우디       4
싱가포르     16
영국       44
이탈리아     72
인도       71
인도네시아    67
일본       72
중국       72
캐나다      50
태국       71
폴란드      63
필리핀      61
호주       47
홍콩       70


In [18]:
trade = pd.read_csv('./data/trade_data_merged.csv')
trade.head()

,연월,연도,월,HS코드,품목명,충격유형,국가코드,국가명,수출금액,수출중량,수입금액,수입중량,무역수지,날짜
0,2020-11,2020,11,720826,열연강판,금리,AU,호주,14624,24620,0,0,14624,2020-11-01
1,2020-12,2020,12,720826,열연강판,금리,AU,호주,14994,24620,0,0,14994,2020-12-01
2,2023-07,2023,7,720826,열연강판,금리,AU,호주,40084,46340,0,0,40084,2023-07-01
3,2023-10,2023,10,720826,열연강판,금리,AU,호주,33264,49500,0,0,33264,2023-10-01
4,2023-11,2023,11,720826,열연강판,금리,AU,호주,22098,29220,0,0,22098,2023-11-01


2. 외국인 투자유치(FDI) 동향

In [7]:
#fdi = pd.read_csv('산업별_외국인_투자_유치_실적_201501_202404.csv', encoding='ANSI', header=None)

#fdi.head()

# 전처리

- 날짜 컬럼 datetime 변환 & 일별 -> 월별 평균

In [36]:
# 각 데이터 날짜 형식 확인
for name, df in [("ex_rate", ex_rate), ("kr_rate", kr_rate),
                 ("us_rate", us_rate),("dubai", dubai)]:
    print(f"{name}: {df['날짜'].iloc[0]} / dtype: {df['날짜'].dtype}")

ex_rate: 2019-01-02 00:00:00 / dtype: datetime64[ns]
kr_rate: 2019-01-01 00:00:00 / dtype: datetime64[ns]
us_rate: 2019-01-01 / dtype: object
dubai: 2019-01-02 00:00:00 / dtype: datetime64[ns]


In [37]:
# 1. 모든 데이터 날짜 컬럼 datetime 변환
ex_rate["날짜"]  = pd.to_datetime(ex_rate["날짜"])                       
kr_rate["날짜"]  = pd.to_datetime(kr_rate["날짜"], format="%Y%m")       
us_rate["날짜"]  = pd.to_datetime(us_rate["날짜"])                      
dubai["날짜"]      = pd.to_datetime(dubai["날짜"], format="%y년%m월%d일")    

In [38]:
# 2. 일별 -> 월별 평균
daily_cols = {"ex_rate": (ex_rate, "원/달러환율"), "dubai": (dubai, "Dubai")}

def to_monthly(df, cols):
    if isinstance(cols, list):
        return df.groupby(df["날짜"].dt.to_period("M"))[cols].mean().reset_index()
    return df.groupby(df["날짜"].dt.to_period("M"))[cols].mean().reset_index()

ex_rate_m = to_monthly(ex_rate, "원/달러환율")
dubai_m     = to_monthly(dubai, "Dubai")

# 월별 데이터도 Period로 통일
kr_rate["날짜"] = kr_rate["날짜"].dt.to_period("M")
us_rate["날짜"] = us_rate["날짜"].dt.to_period("M")

In [40]:
# 3. 병합
X_df = (ex_rate_m
         .merge(kr_rate,          on="날짜", how="left")
         .merge(us_rate,          on="날짜", how="left")
         .merge(dubai_m[["날짜", "Dubai"]], on="날짜", how="left"))

In [41]:
# 4. 분석 기간 필터링
X_df = X_df[
    (X_df["날짜"] >= pd.Period("2019-01", "M")) &
    (X_df["날짜"] <= pd.Period("2024-12", "M"))
].reset_index(drop=True)

print(X_df.shape)   # (72, 5)
print(X_df.head(5))
print(X_df.isnull().sum())  # 결측치 확인

(72, 5)
        날짜       원/달러환율  한국금리  미국금리      Dubai
0  2019-01  1122.000000  1.75  2.40  59.087273
1  2019-02  1122.447059  1.75  2.40  58.128500
2  2019-03  1130.720000  1.75  2.41  66.941905
3  2019-04  1140.950000  1.75  2.42  70.944762
4  2019-05  1183.290476  1.75  2.39  63.350435
날짜        0
원/달러환율    0
한국금리      0
미국금리      0
Dubai     0
dtype: int64


### 단위
- 원/달러 환율: 원(₩) ex. 1,122원
- 한국금리: %
- 미국금리: %
- 두바이유: 달러/배럴 ex. 59.09 달러

In [42]:
X_df.to_csv('독립변수.csv')

### FDI 데이터 Wide -> Long 변환

In [102]:
import numpy as np

# 분기·항목 컬럼명 조합
quarters = fdi.iloc[0, 2:].tolist()
items = fdi.iloc[1, 2:].tolist()

filled_q, last = [], ''
for q in quarters:
    if str(q) != 'nan' and q != last:
        last = q
    filled_q.append(last)

col_names = ['산업대분류', '산업소분류'] + [f'{q}_{it}' for q, it in zip(filled_q, items)]
data = fdi.iloc[2:].copy()
data.columns = col_names
data = data.reset_index(drop = True)

In [103]:
# Wide -> Long 변환
fdi_long = data.melt(id_vars=['산업대분류', '산업소분류'], var_name='분기_항목', value_name='값')
fdi_long['연도'] = fdi_long['분기_항목'].str.extract(r'(\d{4})')
fdi_long['분기'] = fdi_long['분기_항목'].str.extract(r'(\d)/4')
fdi_long['항목'] = fdi_long['분기_항목'].str.extract(r'_(.+)')
fdi_long = fdi_long.drop(columns='분기_항목')

# 신고금액만 추출, 소계 행만
fdi = (fdi_long[fdi_long['항목'] == '신고금액 (천불)']
       .drop(columns='항목')
       .rename(columns={'값': '신고금액(천불)'})
       .reset_index(drop=True))

fdi['신고금액(천불)'] = pd.to_numeric(fdi['신고금액(천불)'].replace('-', np.nan))

print(fdi.shape)
fdi.head(10)

(1320, 5)


,산업대분류,산업소분류,신고금액(천불),연도,분기
0,계,소계,3554801.0,2015,1
1,농·축·수산·광업,소계,291.0,2015,1
2,농·축·수산·광업,농·축·임업,291.0,2015,1
3,농·축·수산·광업,어업,0.0,2015,1
4,농·축·수산·광업,광업,0.0,2015,1
5,제조업,소계,385270.0,2015,1
6,제조업,식품,2340.0,2015,1
7,제조업,섬유·직물·의류,95.0,2015,1
8,제조업,제지·목재,1677.0,2015,1
9,제조업,화공,101019.0,2015,1


In [104]:
print(fdi['산업대분류'].unique())
print(fdi['산업소분류'].unique())

<StringArray>
['계', '농·축·수산·광업', '제조업', '서비스업', '전기·가스·수도·환경정화·건설']
Length: 5, dtype: str
<StringArray>
[          '소계',       '농·축·임업',           '어업',           '광업',
           '식품',     '섬유·직물·의류',        '제지·목재',           '화공',
           '의약',     '비금속 광물제품',    '금속·금속가공제품',    '기계장비·의료정밀',
        '전기·전자',        '운송용기계',         '기타제조',     '도·소매(유통)',
       '숙박·음식점',        '운수·창고',         '정보통신',        '금융·보험',
          '부동산',      '사업지원·임대', '연구개발·전문·과학기술',    '여가·스포츠·오락',
     '공공·기타서비스',        '전기·가스',   '수도·하수·환경정화',         '종합건설',
      '전문직별 공사']
Length: 29, dtype: str


### 수출입 데이터

In [23]:
import pandas as pd

# ================================================
# STEP 1. 데이터 로드
# ================================================

df = pd.read_csv("./data/trade_data_merged.csv", dtype=str)
print(f"원본 행수: {len(df):,}")


# ================================================
# STEP 2. 매핑 테이블 정의
# ================================================

HS_INFO = {
    "854231": ("반도체_메모리",       "환율"),
    "854232": ("반도체_비메모리",     "환율"),
    "854239": ("기타집적회로",        "환율"),
    "851762": ("통신기기부품",        "환율"),
    "851769": ("기타통신기기",        "환율"),
    "852872": ("평판디스플레이",      "환율"),
    "870321": ("승용차_1000cc이하",   "환율"),
    "870322": ("승용차_1000~1500cc", "환율"),
    "870323": ("승용차_1500~3000cc", "환율"),
    "870324": ("승용차_3000cc초과",   "환율"),
    "870840": ("자동차변속기",        "환율"),
    "870850": ("자동차구동축",        "환율"),
    "870899": ("기타자동차부품",      "환율"),
    "901380": ("광학기기",            "환율"),
    "903289": ("계측제어기기",        "환율"),
    "271012": ("경유등유",            "유가"),
    "271019": ("기타석유제품",        "유가"),
    "271111": ("LPG액화",             "유가"),
    "271121": ("LNG액화",             "유가"),
    "290110": ("에틸렌",              "유가"),
    "290121": ("에틸렌기타",          "유가"),
    "290129": ("기타기초유분",        "유가"),
    "390110": ("폴리에틸렌저밀도",    "유가"),
    "390120": ("폴리에틸렌고밀도",    "유가"),
    "390210": ("폴리프로필렌",        "유가"),
    "390410": ("PVC",                 "유가"),
    "390761": ("PET",                 "유가"),
    "330499": ("화장품기타",          "유가"),
    "320419": ("합성염료",            "유가"),
    "381400": ("혼합용제희석제",      "유가"),
    "847989": ("일반기계기타",        "금리"),
    "847950": ("산업용로봇",          "금리"),
    "842951": ("불도저굴착기",        "금리"),
    "842952": ("스크레이퍼",          "금리"),
    "890190": ("선박기타",            "금리"),
    "890120": ("탱커",                "금리"),
    "890130": ("냉동선",              "금리"),
    "720826": ("열연강판",            "금리"),
    "720839": ("냉연강판",            "금리"),
    "730511": ("강관",                "금리"),
    "850760": ("리튬이온이차전지",    "복합"),
    "850780": ("기타이차전지",        "복합"),
    "848620": ("반도체제조장비",      "복합"),
    "848630": ("반도체공정장비",      "복합"),
    "300490": ("의약품기타",          "복합"),
    "300420": ("항생제",              "복합"),
    "880240": ("항공기",              "복합"),
    "880330": ("항공기부품",          "복합"),
    "854140": ("태양전지",            "복합"),
    "902780": ("분석기기",            "복합"),
}

COUNTRY_INFO = {
    "US": "미국",       "CN": "중국",       "VN": "베트남",
    "HK": "홍콩",       "JP": "일본",       "TW": "대만",
    "IN": "인도",       "SG": "싱가포르",   "MY": "말레이시아",
    "AU": "호주",       "DE": "독일",       "NL": "네덜란드",
    "GB": "영국",       "PL": "폴란드",     "IT": "이탈리아",
    "SA": "사우디",     "AE": "UAE",        "MX": "멕시코",
    "CA": "캐나다",     "BR": "브라질",     "TH": "태국",
    "ID": "인도네시아", "PH": "필리핀",     "BD": "방글라데시",
    "RU": "러시아",
}


# ================================================
# STEP 3. 날짜 컬럼 정리
# ================================================

# 날짜 → YYYY-MM Period 형식으로 변환
# 월별 평균 데이터라 월까지만 표시
df["날짜"] = pd.to_datetime(df["날짜"], format="%Y-%m-%d", errors="coerce").dt.to_period("M")

# 불필요한 날짜 컬럼 제거
df = df.drop(columns=["연월", "연도", "월"])

print(f"날짜 컬럼 정리 완료 (YYYY-MM 형식)")


# ================================================
# STEP 4. 숫자형 변환
# ================================================

num_cols = ["수출금액", "수출중량", "수입금액", "수입중량", "무역수지"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

print(f"숫자형 변환 완료")


# ================================================
# STEP 5. 교역 없는 달 0으로 채우기
# ================================================
# 거시지표는 매월 있는데 수출 데이터가 띄엄띄엄 있으면
# 교차 분석 시 불일치 → 교역 없는 달은 0으로 채움

date_range = pd.period_range(
    start=df["날짜"].min(),
    end=df["날짜"].max(),
    freq="M"
)

hs_list      = list(HS_INFO.keys())
country_list = list(COUNTRY_INFO.keys())

print(f"\n0 채우기 시작...")
print(f"  기간: {date_range[0]} ~ {date_range[-1]} ({len(date_range)}개월)")
print(f"  HS코드: {len(hs_list)}개 × 국가: {len(country_list)}개")
print(f"  이론상 최대 행수: {len(date_range) * len(hs_list) * len(country_list):,}")

# 전체 조합 인덱스 생성
full_index = pd.MultiIndex.from_product(
    [date_range, hs_list, country_list],
    names=["날짜", "HS코드", "국가코드"]
)
df_full = pd.DataFrame(index=full_index).reset_index()

# 기존 데이터 병합
df_merged = df_full.merge(
    df[["날짜", "HS코드", "국가코드"] + num_cols],
    on=["날짜", "HS코드", "국가코드"],
    how="left"
)

# 수치 컬럼 0으로 채우기
for col in num_cols:
    df_merged[col] = df_merged[col].fillna(0)

print(f"  0 채우기 완료: {len(df_merged):,}행")


# ================================================
# STEP 6. 품목명·충격유형·국가명 채우기
# ================================================
# 0으로 채운 행은 join이 안 돼서 NaN → 매핑 딕셔너리로 직접 채움

df_merged["품목명"]  = df_merged["HS코드"].map(
    lambda x: HS_INFO.get(str(x), (None, None))[0]
)
df_merged["충격유형"] = df_merged["HS코드"].map(
    lambda x: HS_INFO.get(str(x), (None, None))[1]
)
df_merged["국가명"] = df_merged["국가코드"].map(COUNTRY_INFO)

# NaN 최종 확인
nan_check = df_merged[["품목명", "충격유형", "국가명"]].isnull().sum()
print(f"\nNaN 확인: {nan_check.to_dict()}")


# ================================================
# STEP 7. 정렬 및 저장
# ================================================

df_merged = df_merged.sort_values(
    ["충격유형", "HS코드", "국가코드", "날짜"]
).reset_index(drop=True)

# 컬럼 순서 정리
col_order = ["날짜", "HS코드", "품목명", "충격유형",
             "국가코드", "국가명",
             "수출금액", "수출중량", "수입금액", "수입중량", "무역수지"]
df_merged = df_merged[col_order]

df_merged.to_csv("trade_data_preprocessed.csv", index=False, encoding="utf-8-sig")

print(f"\n{'='*50}")
print(f"✅ 전처리 완료: trade_data_preprocessed.csv")
print(f"   최종 행수: {len(df_merged):,}")
print(f"\n▶ 컬럼 구성:")
for col in df_merged.columns:
    print(f"   {col}: {df_merged[col].dtype}")

print(f"\n▶ 샘플 (열연강판 × 호주 2019년 - 0 채우기 확인):")
sample = df_merged[
    (df_merged["HS코드"] == "720826") &
    (df_merged["국가코드"] == "AU") &
    (df_merged["날짜"].astype(str).str.startswith("2019"))
]
print(sample[["날짜", "수출금액", "수입금액"]].to_string(index=False))

원본 행수: 63,576
날짜 컬럼 정리 완료 (YYYY-MM 형식)
숫자형 변환 완료

0 채우기 시작...
  기간: 2019-01 ~ 2024-12 (72개월)
  HS코드: 50개 × 국가: 25개
  이론상 최대 행수: 90,000
  0 채우기 완료: 90,000행

NaN 확인: {'품목명': 0, '충격유형': 0, '국가명': 0}

✅ 전처리 완료: trade_data_preprocessed.csv
   최종 행수: 90,000

▶ 컬럼 구성:
   날짜: period[M]
   HS코드: object
   품목명: object
   충격유형: object
   국가코드: object
   국가명: object
   수출금액: float64
   수출중량: float64
   수입금액: float64
   수입중량: float64
   무역수지: float64

▶ 샘플 (열연강판 × 호주 2019년 - 0 채우기 확인):
     날짜  수출금액  수입금액
2019-01   0.0   0.0
2019-02   0.0   0.0
2019-03   0.0   0.0
2019-04   0.0   0.0
2019-05   0.0   0.0
2019-06   0.0   0.0
2019-07   0.0   0.0
2019-08   0.0   0.0
2019-09   0.0   0.0
2019-10   0.0   0.0
2019-11   0.0   0.0
2019-12   0.0   0.0
